In [1]:
import numpy as np
import pandas as pd
import pypsa

import sys
from pathlib import Path

sys.path.insert(0, str(Path("src")))

from src import network_builder
from src import util
from util import annualized_capex


In [2]:
data = pd.read_csv("data/clean_data.csv")

In [3]:
snapshots = data.index
nHours = len(snapshots)

load = pd.DataFrame(index=snapshots)
load["reg1"] = data["load_mw"] * .3
load["reg2"] = data["load_mw"] *  .5
load["reg3"] = data["load_mw"] * .7

solar = pd.DataFrame(index=snapshots)
solar["reg1"] = data["ca_pv_cf"]
solar["reg2"] = data["az_pv_cf"]
solar["reg3"] = data["nv_pv_cf"]

wind = pd.DataFrame(index=snapshots)
wind["reg1"] = data["ca_wind_cf"]
wind["reg2"] = data["pnw_wind_cf"]
wind["reg3"] = data["wy_wind_cf"]

In [4]:
total_load_mwh = load.sum().sum()
baseline_emissions = total_load_mwh * 0.4
co2_cap = baseline_emissions * 0.05   # for a 95% reduction


ldes_cost_case = "medium"
transmission_friction = "low"
ldes_duration_hours = 24


ldes_cost_multipliers = {
        "low": 0.70,
        "medium": 1.00,
        "high": 1.40,
    }

ldes_base_capex_per_mw = 1_800_000
    transmission_base_capex_per_mw = 400_000



In [6]:
n = network_builder.build_network(snapshots, load, wind, solar, co2_cap, ldes_cost_case, transmission_friction, ldes_duration_hours)

In [7]:
RESOLUTION = 1
n.set_snapshots(n.snapshots[::RESOLUTION])
n.snapshot_weightings.loc[:, :] = RESOLUTION

n.optimize()

/var/folders/t_/32_5y8z906v9c0l8qphz6hx00000gn/T/ipykernel_68507/243138191.py:5: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

Index(['reg1', 'reg2', 'reg3'], dtype='object', name='name')
Index(['tx_reg1_reg2', 'tx_reg1_reg3', 'tx_reg2_reg3'], dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|█████████████| 8/8 [00:00<00:00, 572.08it/s]
INFO:linopy.io: Writing time: 0.29s


Running HiGHS 1.13.1 (git hash: n/a): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-kox2ect8 has 603907 rows; 262578 cols; 1246087 nonzeros
Coefficient ranges:
  Matrix  [1e-03, 2e+01]
  Cost    [1e+02, 2e+05]
  Bound   [0e+00, 0e+00]
  RHS     [5e+03, 7e+06]
Presolving model
327110 rows, 248359 cols, 955071 nonzeros  0s
Dependent equations search running on 78768 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.01s (limit = 1000.00s)
327110 rows, 248359 cols, 955071 nonzeros  0s
Presolve reductions: rows 327110(-276797); columns 248359(-14219); nonzeros 955071(-291016) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 26256(2.59738e+08) 0.4s
      32390     7.9024420119e+09 Pr: 105972(5.82746e+09); Du: 0(1.54519e-06) 6.2s
      38505     7.9026372753e+09 Pr: 106647(9.48671e+09); Du: 0(1.45445e-06) 12.0s
      44050     7.902

INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 262578 primals, 603907 duals
Objective: 3.81e+10
Solver model: available
Solver message: Optimal




Model name          : linopy-problem-kox2ect8
Model status        : Optimal
Simplex   iterations: 351379
Objective value     :  3.8100505607e+10
P-D objective error :  7.6493176783e-14
HiGHS run time      :       1136.47


INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-ext-p-lower, Generator-ext-p-upper, Link-ext-p-lower, Link-ext-p-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance were not assigned to the network.


('ok', 'optimal')

In [8]:
n.generators

,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_nom_set,p_min_pu,...,min_up_time,min_down_time,up_time_before,down_time_before,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,weight,p_nom_opt
name,,,,,,,,,,,,,,,,,,,,,
wind_reg1,reg1,Slack,,0.0,0.0,True,0.0,inf,NaN,0.0,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,53353.933234
solar_reg1,reg1,PQ,,0.0,0.0,True,0.0,inf,NaN,0.0,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,69733.676497
backup_reg1,reg1,PQ,,0.0,0.0,True,0.0,inf,NaN,0.0,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,139.605041
wind_reg2,reg2,Slack,,0.0,0.0,True,0.0,inf,NaN,0.0,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,-0.000000
solar_reg2,reg2,PQ,,0.0,0.0,True,0.0,inf,NaN,0.0,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,29543.725798
backup_reg2,reg2,PQ,,0.0,0.0,True,0.0,inf,NaN,0.0,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,5942.521706
wind_reg3,reg3,Slack,,0.0,0.0,True,0.0,inf,NaN,0.0,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,26301.400149
solar_reg3,reg3,PQ,,0.0,0.0,True,0.0,inf,NaN,0.0,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,20505.239449
backup_reg3,reg3,PQ,,0.0,0.0,True,0.0,inf,NaN,0.0,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,8477.603968


In [9]:
n.links

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,shut_down_cost,min_up_time,min_down_time,up_time_before,down_time_before,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,p_nom_opt
name,,,,,,,,,,,,,,,,,,,,,
tx_reg1_reg2,reg1,reg2,,AC,1.0,True,0,inf,0.0,0.0,...,0.0,0,0,1,0,NaN,NaN,NaN,NaN,11487.598290
tx_reg1_reg3,reg1,reg3,,AC,1.0,True,0,inf,0.0,0.0,...,0.0,0,0,1,0,NaN,NaN,NaN,NaN,12514.054959
tx_reg2_reg3,reg2,reg3,,AC,1.0,True,0,inf,0.0,0.0,...,0.0,0,0,1,0,NaN,NaN,NaN,NaN,367.388113


In [12]:
n.storage_units

,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_nom_set,p_min_pu,...,state_of_charge_initial_per_period,state_of_charge_set,cyclic_state_of_charge,cyclic_state_of_charge_per_period,max_hours,efficiency_store,efficiency_dispatch,standing_loss,inflow,p_nom_opt
name,,,,,,,,,,,,,,,,,,,,,
battery4h_reg1,reg1,PQ,,0.0,0.0,True,0.0,inf,NaN,-1.0,...,False,NaN,True,False,4.0,0.92,0.92,0.0,0.0,3796.783708
ldes_24h_reg1,reg1,PQ,,0.0,0.0,True,0.0,inf,NaN,-1.0,...,False,NaN,True,False,24.0,0.80,0.80,0.0,0.0,23961.936407
battery4h_reg2,reg2,PQ,,0.0,0.0,True,0.0,inf,NaN,-1.0,...,False,NaN,True,False,4.0,0.92,0.92,0.0,0.0,5586.585948
ldes_24h_reg2,reg2,PQ,,0.0,0.0,True,0.0,inf,NaN,-1.0,...,False,NaN,True,False,24.0,0.80,0.80,0.0,0.0,3433.321808
battery4h_reg3,reg3,PQ,,0.0,0.0,True,0.0,inf,NaN,-1.0,...,False,NaN,True,False,4.0,0.92,0.92,0.0,0.0,-0.000000
ldes_24h_reg3,reg3,PQ,,0.0,0.0,True,0.0,inf,NaN,-1.0,...,False,NaN,True,False,24.0,0.80,0.80,0.0,0.0,7606.960843
